# Use Case: Simulating a Customer Interview

This notebook demonstrates how TinyTroupe can be used to simulate an interview with a specific customer persona. This is valuable for:
- Understanding potential customer pain points.
- Gathering synthetic feedback on product ideas or concepts.
- Refining marketing messages.
- Training for user research and interview techniques.

The process involves:
1. Defining a target customer profile using `TinyPersonFactory` by providing context (their industry/company) and specific particularities.
2. Generating the customer agent.
3. (Recommended) Validating the generated agent's persona against key expectations to ensure it's suitable for the interview using `TinyPersonValidator`.
4. Conducting a simulated interview by sending a series of questions (as stimuli) to the agent.
5. Analyzing the agent's responses to gather insights.

## 1. Setup and Imports

Import necessary classes: `TinyPerson` for agent representation, `TinyPersonFactory` for creating our customer agent, `TinyPersonValidator` for ensuring the agent aligns with our target profile, and `TinyWorld` to provide a (simple) context for the interaction if needed, though for a one-on-one interview, the world's role might be minimal.

In [ ]:
import json
import sys
# If running from 'examples/use_cases/', this adds the parent directory of 'examples' (project root) to Python path.
sys.path.insert(0, '../..') 

import tinytroupe # Initializes configuration, logging, etc.
from tinytroupe.agent import TinyPerson
from tinytroupe.factory import TinyPersonFactory
from tinytroupe.validation import TinyPersonValidator
# TinyWorld might not be strictly necessary if the interview is very direct,
# but it's good practice to have an environment for agents.
from tinytroupe.environment import TinyWorld 

# ResultsReducer and control are not used in this specific notebook's flow.
# from tinytroupe.extraction import ResultsReducer
# import tinytroupe.control as control
import textwrap # For pretty printing long strings

## 2. Define and Generate the Customer Persona

We'll create a persona for a Vice-President of Product Innovation at a large, traditional bank that's facing competition from agile fintech companies. This sets the stage for an interview about their challenges and needs.

In [ ]:
# Context for the TinyPersonFactory: the company/industry environment
factory_context = "The customer works at one of the largest banks in Brazil, which is characterized by significant bureaucracy and reliance on legacy systems. There is mounting pressure to innovate due to fintech competition."
factory = TinyPersonFactory(factory_context)

# Specific particularities for the customer persona
customer_particularities = (
    "A Vice-President of Product Innovation. This executive has a degree in engineering and an MBA in finance. "
    "They are under considerable pressure from the board of directors to develop strategies and products "
    "that can effectively counter the competition from nimble fintech startups."
)

# Generate the customer agent
customer = factory.generate_person(customer_particularities)

if customer:
    print(f"Generated customer agent: {customer.name}")
else:
    print("Customer agent generation failed. Check logs for details.")

### 2.1. Inspect the Generated Customer's Minibio

Let's get a brief overview of the generated persona.

In [ ]:
if customer:
    print(textwrap.fill(customer.minibio(), width=100))

### 2.2. Validate the Customer Persona (Recommended)

Before conducting the interview, it's good practice to validate that the generated agent aligns with the key characteristics of our target interviewee. This ensures the feedback gathered will be relevant.

In [ ]:
customer_expectations =\
    """
    The individual should be:
    - A high-level executive (e.g., Vice-President) in a large, traditional Brazilian bank.
    - Possess an engineering background and an MBA in finance.
    - Actively involved in product innovation.
    - Acutely aware of and concerned about competition from fintechs.
    - Likely in their 40s or 50s.
    - Articulate and strategic in their thinking.
    
    Regarding their professional context:
    - Works in an environment with bureaucracy and legacy systems.
    - Faces pressure from the board/upper management regarding innovation and competition.
    """
if customer:
    customer_score, customer_justification = TinyPersonValidator.validate_person(
        person=customer, 
        expectations=customer_expectations, 
        include_agent_spec=False, # Base validation on interaction, not just the initial spec
        max_content_length=None
    )
    print(f"Customer Persona Validation Score: {customer_score}")
    print("\nJustification:")
    print(textwrap.fill(customer_justification, width=100))
else:
    print("Skipping validation as customer agent generation failed.")

## 3. Conduct the Simulated Interview

Now, we'll simulate an interview by sending a series of questions to our customer agent. We'll use `listen_and_act()` for each question to get the agent's response. It's helpful to give the agent an initial thought to set the context for the interview.

In [ ]:
if customer:
    # Set the initial context for the customer agent
    customer.think("I am now talking to a business and technology consultant who is here to help me identify and solve my key professional problems and challenges regarding product innovation in my bank.")
    print(f"Set initial thought for {customer.name}. Starting interview...\n")
    
    # --- Interview Question 1 --- 
    question1 = "What would you say are your main problems today regarding product innovation and competing with fintechs? Please be as specific as possible."
    print(f"YOU: {question1}")
    customer.listen_and_act(question1, max_content_length=3000) # max_content_length for display
    # The agent's response will be printed to the console by default.
else:
    print("Cannot start interview as customer agent was not generated.")

### 3.1. Follow-up Questions

Based on the initial response, we can ask follow-up questions to delve deeper into specific areas.

In [ ]:
if customer:
    # --- Interview Question 2 --- 
    question2 = "Can you elaborate on the specific pressures from fintechs and how they impact your current product strategy?"
    print(f"\nYOU: {question2}")
    customer.listen_and_act(question2, max_content_length=3000)
else:
    print("Skipping question 2.")

In [ ]:
if customer:
    # --- Interview Question 3 --- 
    question3 = "If you could improve in one key aspect of your innovation process to better compete, what would that be and why?"
    print(f"\nYOU: {question3}")
    customer.listen_and_act(question3, max_content_length=3000)
else:
    print("Skipping question 3.")

### 3.2. Probing for Solutions/Project Directions

In [ ]:
if customer:
    # --- Interview Question 4 --- 
    question4 = "Thank you for those insights. Based on what you've said, if we were to start a project to address these challenges, what specific area or problem should we focus on first to make the most impact? Please give me some details on that."
    print(f"\nYOU: {question4}")
    customer.listen_and_act(question4, max_content_length=3000)
    print("\nInterview concluded.")
else:
    print("Skipping question 4.")

## 4. Reviewing the Interaction Log

After the interview, you can review the agent's full interaction history (including its thoughts if you enabled detailed logging or used `think` explicitly) to extract key insights, pain points, and potential opportunities. TinyTroupe prints interactions to the console by default. For more structured analysis, you would typically use the `ResultsExtractor` (as shown in other examples) or custom logging.

In [ ]:
if customer:
    print(f"\n--- Full Interaction Log for {customer.name} ---")
    customer.pp_current_interactions(max_content_length=500) # Truncate long content for readability
else:
    print("No customer agent to display logs for.")

This simulated interview allows for exploration of a customer's mindset without the logistical overhead of real interviews, making it a useful tool for initial research, hypothesis testing, or preparing for actual customer engagements.